# Run the HIF / CICIDS2017 comparison on Google Colab

This notebook clones the repository, installs the pinned dependencies,
downloads the dataset and runs the full comparison. It is meant to run on
Colab so you do not have to run the heavy pipeline on your own machine.

Recommended: Runtime -> Change runtime type -> High-RAM (the dataset has
millions of rows). A GPU is not needed; the models run on CPU.

## 1. Clone the repository
Edit REPO_URL if you forked it or pushed it elsewhere.

In [ ]:
REPO_URL = "https://github.com/Dicotomico23/hif-cicids2017.git"
import os
repo = REPO_URL.rstrip('/').split('/')[-1].replace('.git', '')
if not os.path.isdir(repo):
    !git clone $REPO_URL
%cd $repo
!git log --oneline -1

## 2. Install dependencies

In [ ]:
!pip install -q -r requirements.txt
!pip install -q -e .

## 3. Get the dataset

First choice: `data/download.py` pulls the archived copy (Zenodo / GitHub
Release) and verifies its checksum, no Kaggle account needed.

If the archive URLs are not set yet, it falls back to Kaggle. To enable the
Kaggle fallback, upload your `kaggle.json` token when prompted (Kaggle ->
Settings -> API -> Create New Token).

In [ ]:
# Optional: provide Kaggle credentials for the fallback path.
import os
if not os.path.exists(os.path.expanduser('~/.kaggle/kaggle.json')):
    try:
        from google.colab import files
        print('Upload kaggle.json (or skip this cell if the archive URLs are set):')
        up = files.upload()
        if up:
            os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
            !mv kaggle.json ~/.kaggle/kaggle.json
            !chmod 600 ~/.kaggle/kaggle.json
    except Exception as e:
        print('Skipped Kaggle upload:', e)

In [ ]:
!python data/download.py

## 4. Run the full comparison

Writes the results table and figures to `results/`. To do a quick partial
run first, add `--nrows 100000`.

In [ ]:
!python reproduce/run_comparison.py --output results

## 5. Show the results

In [ ]:
import pandas as pd
from IPython.display import Image, display
print(pd.read_csv('results/table5_results.csv').to_string(index=False))
for fig in ['fig_radar_metrics', 'fig_hif_confusion_matrix', 'fig_bar_balanced_acc', 'fig_bar_precision']:
    display(Image('results/figures/%s.png' % fig))